# PyPSA Network Comparative Analysis

This notebook provides a modular framework for comparing multiple solved PyPSA networks. 
It includes helper functions for:
- Loading networks and building metadata tables
- Safely extracting optimized vs. installed capacities
- Computing snapshot weighting in hours
- Checking component availability in networks

**Purpose:** Comparative analysis of solved network runs (not optimization)

## Section 1: Imports and Configuration

In [1]:
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, Tuple, Optional, List, Union
import warnings

# Configure display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Suppress warnings for cleaner output (optional)
warnings.filterwarnings('ignore', category=FutureWarning)

In [2]:
# Configuration constants (centralized)
OUTPUT_DIR = Path("/home/lucakristin/Desktop/my_pypsa/pypsa-eur/summary_csv")
CONGESTION_THRESHOLD = 60.0
TOP_N_LINES = 5
TOP_N_BUSES = 5
SHEDDING_CARRIER_REGEX = "load"  # used with pandas .str.contains
SNAPSHOT_HOURS = 1.0  # assume uniform hourly snapshots

# Verbosity control
VERBOSE = True

# Ensure output directory exists
try:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
except Exception:
    pass

In [3]:
# Consolidated component-check helpers and snapshot-hours helper
def has_generators(n: pypsa.Network) -> bool:
    return not getattr(n, 'generators', pd.DataFrame()).empty

def has_lines(n: pypsa.Network) -> bool:
    return not getattr(n, 'lines', pd.DataFrame()).empty

def has_loads(n: pypsa.Network) -> bool:
    return not getattr(n, 'loads', pd.DataFrame()).empty

def has_buses(n: pypsa.Network) -> bool:
    return not getattr(n, 'buses', pd.DataFrame()).empty

def has_links(n: pypsa.Network) -> bool:
    return not getattr(n, 'links', pd.DataFrame()).empty

def has_stores(n: pypsa.Network) -> bool:
    return not getattr(n, 'stores', pd.DataFrame()).empty

def snapshot_hours(network: pypsa.Network) -> pd.Series:
    """Return a pd.Series of snapshot hours aligned to network.snapshots."""
    try:
        return pd.Series(SNAPSHOT_HOURS, index=network.snapshots)
    except Exception:
        # Fallback: return ones with length equal to snapshots
        return pd.Series(np.ones(len(getattr(network, 'snapshots', []))) * SNAPSHOT_HOURS,
                         index=getattr(network, 'snapshots', []))

## Section 2: Define Results Folder and Network Dictionary

The solved networks are stored in a nested directory structure:
```
/home/lucakristin/Desktop/my_pypsa/pypsa-eur/results/
├── germany_base2_notwindy/
│   └── networks/
│       └── base_s_450_elec_.nc
├── germany_baseTP_windvariability/
│   └── networks/
│       └── base_s_450_elec_.nc
└── ...
```

Networks are loaded into a dictionary with tuple keys: `(scenario_name, wind_condition, transmission_setting)`

In [4]:
# Define the base results directory
RESULTS_DIR = Path("/home/lucakristin/Desktop/my_pypsa/pypsa-eur/results")
NETWORK_FILENAME = "base_s_450_elec_.nc"
NETWORKS_SUBFOLDER = "networks"

def parse_network_directory_name(dir_name: str) -> Optional[Tuple[str, str, str]]:
    """
    Parse a network directory name to extract metadata.
    
    Assumes directory names follow the pattern: germany_{scenario}_{wind_condition}
    
    Parameters
    ----------
    dir_name : str
        Directory name (e.g., 'germany_base2_notwindy')
    
    Returns
    -------
    Tuple[str, str, str] or None
        (scenario_name, wind_condition, transmission_setting) or None if invalid
        
    Examples
    --------
    >>> parse_network_directory_name('germany_base2_notwindy')
    ('base2', 'notwindy', 'fixed')
    >>> parse_network_directory_name('germany_scenario-ext_windy')
    ('scenario-ext', 'windy', 'extendable')
    """
    if not dir_name.startswith("germany_"):
        return None
    
    # Remove 'germany_' prefix
    remainder = dir_name[8:]  # len("germany_") = 8
    
    # Wind condition is the last component (notwindy, windy, windvariability)
    wind_conditions = ['notwindy', 'windy', 'windvariability']
    wind_condition = None
    
    for wc in wind_conditions:
        if remainder.endswith(wc):
            wind_condition = wc
            scenario_name = remainder[:-len(wc)-1]  # Remove wind condition and trailing '_'
            break
    
    if wind_condition is None:
        return None
    
    # Define transmission settings based on scenario name
    transmission_map = {
        'base': 'fixed',
        'base2': 'fixed',
        'scenario1': 'fixed',
        'scenario2': 'fixed',
        'scenario-ext': 'extendable + projects',
        'baseTP': 'projects',
        'base-ext': 'extendable'
    }
    
    transmission_setting = transmission_map.get(scenario_name, 'unknown')
    
    return (scenario_name, wind_condition, transmission_setting)


def load_networks_from_results(
    results_dir: Path = RESULTS_DIR,
    network_filename: str = NETWORK_FILENAME,
    networks_subfolder: str = NETWORKS_SUBFOLDER
) -> Dict[Tuple[str, str, str], pypsa.Network]:
    """
    Load all available solved networks from the results directory.
    
    Parameters
    ----------
    results_dir : Path
        Base results directory containing scenario subdirectories
    network_filename : str
        Name of the network file (default: 'base_s_450_elec_.nc')
    networks_subfolder : str
        Subfolder name containing network files (default: 'networks')
    
    Returns
    -------
    Dict[Tuple[str, str, str], pypsa.Network]
        Dictionary with tuple keys (scenario_name, wind_condition, transmission_setting)
    """
    networks = {}
    
    for scenario_dir in sorted(results_dir.iterdir()):
        if not scenario_dir.is_dir():
            continue
        
        # Parse the directory name
        metadata = parse_network_directory_name(scenario_dir.name)
        if metadata is None:
            continue
        
        # Check if network file exists
        network_path = scenario_dir / networks_subfolder / network_filename
        if not network_path.exists():
            print(f"[WARNING] Network file not found: {network_path}")
            continue
        
        try:
            network = pypsa.Network()
            network.import_from_netcdf(str(network_path))
            networks[metadata] = network
        except Exception as e:
            print(f"[ERROR] {e}")
    return networks

# Load networks
networks = load_networks_from_results()
print(f"Available networks: {len(networks)}")


INFO:pypsa.network.io:New version 1.2.1 available! (Current: 1.1.2)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.2.1 available! (Current: 1.1.2)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.2.1 available! (Current: 1.1.2)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.2.1 available! (Current: 1.1.2)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.2.1 available! (Current:

Available networks: 18


## Section 3: Build Network Run Metadata DataFrame

In [6]:
def build_metadata_dataframe(
    networks: Dict[Tuple[str, str, str], pypsa.Network]
) -> pd.DataFrame:
    """
    Build a metadata DataFrame from the networks dictionary.
    
    Parameters
    ----------
    networks : Dict[Tuple[str, str, str], pypsa.Network]
        Dictionary of loaded networks with tuple keys
        (scenario_name, wind_condition, transmission_setting)
    
    Returns
    -------
    pd.DataFrame
        Metadata for all runs with columns:
        - scenario_name
        - wind_condition
        - transmission_setting
        - n_snapshots
        - n_buses
        - n_generators
        - n_lines
        - n_loads
        - n_links
        - n_stores
        - objective
    """
    metadata = []
    
    for (scenario_name, wind_condition, transmission_setting), network in networks.items():
        # Get network statistics
        n_snapshots = len(network.snapshots)
        n_buses = len(network.buses)
        n_generators = len(network.generators)
        n_lines = len(network.lines)
        n_loads = len(network.loads)
        n_links = len(network.links)
        n_stores = len(network.stores)
        
        # Get objective value if network was solved
        objective = getattr(network, 'objective', np.nan)
        
        metadata.append({
            'scenario_name': scenario_name,
            'wind_condition': wind_condition,
            'transmission_setting': transmission_setting,
            'n_snapshots': n_snapshots,
            'n_buses': n_buses,
            'n_generators': n_generators,
            'n_lines': n_lines,
            'n_loads': n_loads,
            'n_links': n_links,
            'n_stores': n_stores,
            'objective': objective,
        })
    
    return pd.DataFrame(metadata)


# Build and display metadata DataFrame
runs_metadata = build_metadata_dataframe(networks)
print("Network Run Metadata:")
print(runs_metadata.to_string())
print(f"\nTotal runs: {len(runs_metadata)}")


Network Run Metadata:
   scenario_name   wind_condition   transmission_setting  n_snapshots  n_buses  n_generators  n_lines  n_loads  n_links  n_stores     objective
0       base-ext         notwindy             extendable           24     1350          4592      628      450     1800       900  1.938391e+07
1       base-ext  windvariability             extendable           24     1350          4592      628      450     1800       900  9.237992e+06
2       base-ext            windy             extendable           24     1350          4592      628      450     1800       900  1.316651e+05
3          base2         notwindy                  fixed           24     1350          4592      628      450     1800       900  2.091066e+07
4          base2  windvariability                  fixed           24     1350          4592      628      450     1800       900  1.827415e+07
5          base2            windy                  fixed           24     1350          4592      628      450    

## Section 4: Safely Retrieve Optimized or Installed Capacities

Helper functions to extract capacity data, with fallback logic for when optimized values are not available.
Remember: `s_nom` is the original nominal value, `s_nom_opt` is the optimized value after solving.

In [7]:
def get_generator_capacities(
    network: pypsa.Network,
    use_optimized: bool = True
) -> pd.DataFrame:
    """
    Extract generator capacities with fallback logic.
    
    Returns p_nom_opt if available and use_optimized=True, otherwise p_nom.
    
    Parameters
    ----------
    network : pypsa.Network
        PyPSA network object
    use_optimized : bool
        If True, prefer p_nom_opt; if False, use p_nom
    
    Returns
    -------
    pd.DataFrame
        Generator data with columns: ['p_nom', 'p_nom_opt', 'capacity_used']
        The 'capacity_used' column contains the effective capacity being used.
    """
    if network.generators.empty:
        return pd.DataFrame()
    
    capacity_df = network.generators[['p_nom']].copy()
    
    # Add optimized capacity if available
    if 'p_nom_opt' in network.generators.columns:
        capacity_df['p_nom_opt'] = network.generators['p_nom_opt']
    else:
        capacity_df['p_nom_opt'] = np.nan
    
    # Determine which capacity is being used
    if use_optimized and 'p_nom_opt' in network.generators.columns:
        capacity_df['capacity_used'] = network.generators['p_nom_opt']
    else:
        capacity_df['capacity_used'] = network.generators['p_nom']
    
    return capacity_df


def get_line_capacities(
    network: pypsa.Network,
    use_optimized: bool = True
) -> pd.DataFrame:
    """
    Extract line capacities with fallback logic.
    
    Returns s_nom_opt if available and use_optimized=True, otherwise s_nom.
    
    Parameters
    ----------
    network : pypsa.Network
        PyPSA network object
    use_optimized : bool
        If True, prefer s_nom_opt; if False, use s_nom
    
    Returns
    -------
    pd.DataFrame
        Line data with columns: ['s_nom', 's_nom_opt', 'capacity_used']
        The 'capacity_used' column contains the effective capacity being used.
    """
    if network.lines.empty:
        return pd.DataFrame()
    
    capacity_df = network.lines[['s_nom']].copy()
    
    # Add optimized capacity if available
    if 's_nom_opt' in network.lines.columns:
        capacity_df['s_nom_opt'] = network.lines['s_nom_opt']
    else:
        capacity_df['s_nom_opt'] = np.nan
    
    # Determine which capacity is being used
    if use_optimized and 's_nom_opt' in network.lines.columns:
        capacity_df['capacity_used'] = network.lines['s_nom_opt']
    else:
        capacity_df['capacity_used'] = network.lines['s_nom']
    
    return capacity_df


def get_link_capacities(
    network: pypsa.Network,
    use_optimized: bool = True
) -> pd.DataFrame:
    """
    Extract link capacities with fallback logic.
    
    Returns p_nom_opt if available and use_optimized=True, otherwise p_nom.
    
    Parameters
    ----------
    network : pypsa.Network
        PyPSA network object
    use_optimized : bool
        If True, prefer p_nom_opt; if False, use p_nom
    
    Returns
    -------
    pd.DataFrame
        Link data with columns: ['p_nom', 'p_nom_opt', 'capacity_used']
    """
    if network.links.empty:
        return pd.DataFrame()
    
    capacity_df = network.links[['p_nom']].copy()
    
    # Add optimized capacity if available
    if 'p_nom_opt' in network.links.columns:
        capacity_df['p_nom_opt'] = network.links['p_nom_opt']
    else:
        capacity_df['p_nom_opt'] = np.nan
    
    # Determine which capacity is being used
    if use_optimized and 'p_nom_opt' in network.links.columns:
        capacity_df['capacity_used'] = network.links['p_nom_opt']
    else:
        capacity_df['capacity_used'] = network.links['p_nom']
    
    return capacity_df


def get_store_capacities(
    network: pypsa.Network,
    use_optimized: bool = True
) -> pd.DataFrame:
    """
    Extract store (energy storage) capacities with fallback logic.
    
    Returns e_nom_opt if available and use_optimized=True, otherwise e_nom.
    
    Parameters
    ----------
    network : pypsa.Network
        PyPSA network object
    use_optimized : bool
        If True, prefer e_nom_opt; if False, use e_nom
    
    Returns
    -------
    pd.DataFrame
        Store data with columns: ['e_nom', 'e_nom_opt', 'capacity_used']
    """
    if network.stores.empty:
        return pd.DataFrame()
    
    capacity_df = network.stores[['e_nom']].copy()
    
    # Add optimized capacity if available
    if 'e_nom_opt' in network.stores.columns:
        capacity_df['e_nom_opt'] = network.stores['e_nom_opt']
    else:
        capacity_df['e_nom_opt'] = np.nan
    
    # Determine which capacity is being used
    if use_optimized and 'e_nom_opt' in network.stores.columns:
        capacity_df['capacity_used'] = network.stores['e_nom_opt']
    else:
        capacity_df['capacity_used'] = network.stores['e_nom']
    
    return capacity_df


# Example: Extract generator capacities from first network
if networks:
    first_network = next(iter(networks.values()))
    gen_caps = get_generator_capacities(first_network, use_optimized=True)
    print("Generator Capacities (first 5 rows):")
    print(gen_caps.head() if not gen_caps.empty else "No generators found")


Generator Capacities (first 5 rows):
                   p_nom  p_nom_opt  capacity_used
name                                              
DE0 0 OCGT      13.57003   13.57003       13.57003
DE0 0 biomass  153.65800  153.65800      153.65800
DE0 0 oil        4.16800    4.16800        4.16800
DE0 0 waste      0.59000    0.59000        0.59000
DE0 1 OCGT       1.68400    1.68400        1.68400


## Section 7: Compute Network Summaries for Comparative Analysis

In [9]:
# Use consolidated component helper functions defined above: has_loads, has_generators, has_lines, etc.

def compute_network_summary(
    network: pypsa.Network,
    scenario_name: str,
    wind_condition: str,
    transmission_setting: str
) -> Dict[str, Union[str, float, int]]:
    """
    Compute a one-row summary for a solved PyPSA network.

    Robust to missing data and various network configurations.

    Parameters
    ----------
    network : pypsa.Network
        Solved PyPSA network
    scenario_name : str
        Name of the scenario
    wind_condition : str
        Wind condition (e.g., 'windy', 'notwindy', 'windvariability')
    transmission_setting : str
        Transmission setting (e.g., 'fixed', 'extended')

    Returns
    -------
    Dict[str, Union[str, float, int]]
        Dictionary with one-row summary containing:
        - Scenario metadata (scenario, wind_condition, transmission_setting)
        - Load statistics (load_served_MWh, load_shedding_MWh)
        - Marginal price statistics (avg_marginal_price, max_marginal_price)
        - Emissions (total_co2_emissions)
        - Line loading statistics (avg_line_loading_pct, max_line_loading_pct, 
          lines_above_60pct, lines_above_70pct)
    """
    summary = {
        'scenario': scenario_name,
        'wind_condition': wind_condition,
        'transmission_setting': transmission_setting,
    }

    # Get snapshot weights for proper weighting (uniform hours)
    snapshot_weights = snapshot_hours(network)

    # =====================================================================
    # LOAD STATISTICS
    # =====================================================================

    # Total demand input in MWh (loads_t.p_set is the input set-point time series)
    if has_loads(network) and hasattr(network, 'loads_t') and hasattr(network.loads_t, 'p_set') and not network.loads_t.p_set.empty:
        total_load = (network.loads_t.p_set.sum(axis=1) * snapshot_weights).sum()
    else:
        total_load = np.nan
    summary['load_served_MWh'] = total_load

    # Load shedding in MWh (if available)
    load_shedding = np.nan
    if 'Load' in network.generators.carrier.values:
        # Try to find load-shedding generators
        load_shed_gens = network.generators[network.generators.carrier == 'Load']
        if len(load_shed_gens) > 0:
            if hasattr(network, 'generators_t') and hasattr(network.generators_t, 'p'):
                shed_output = network.generators_t.p[load_shed_gens.index].sum(axis=1)
                load_shedding = (shed_output * snapshot_weights).sum()
    summary['load_shedding_MWh'] = load_shedding

    # =====================================================================
    # MARGINAL PRICE STATISTICS
    # =====================================================================

    avg_marginal_price = np.nan
    max_marginal_price = np.nan

    if hasattr(network, 'buses_t') and hasattr(network.buses_t, 'marginal_price'):
        marginal_prices = network.buses_t.marginal_price
        if not marginal_prices.empty:
            # Flatten and remove NaN values
            flat_prices = marginal_prices.values.flatten()
            flat_prices = flat_prices[~np.isnan(flat_prices)]

            if len(flat_prices) > 0:
                # Weight by snapshot for proper average
                # Create weighted average: for each snapshot, average across buses, then weight snapshots
                avg_by_snapshot = marginal_prices.mean(axis=1)
                avg_marginal_price = (avg_by_snapshot * snapshot_weights).sum() / snapshot_weights.sum()
                max_marginal_price = flat_prices.max()

    summary['avg_marginal_price'] = avg_marginal_price
    summary['max_marginal_price'] = max_marginal_price

    # =====================================================================
    # CO2 EMISSIONS
    # =====================================================================

    total_co2 = np.nan

    # Try to get CO2 emissions using carrier-level emissions (carriers.co2_emissions)
    if has_generators(network) and hasattr(network, 'generators_t'):
        if hasattr(network.generators_t, 'p') and not network.generators_t.p.empty:
            gen_output = network.generators_t.p  # snapshots x generators

            # Prefer carrier-level emissions stored in network.carriers['co2_emissions']
            carriers = getattr(network, 'carriers', pd.DataFrame())
            if not carriers.empty and 'co2_emissions' in carriers.columns:
                try:
                    emissions_by_carrier = carriers['co2_emissions'].fillna(0)
                    # Map carrier emissions to generators via generators.carrier
                    emissions = network.generators['carrier'].map(emissions_by_carrier).fillna(0)
                    # Calculate hourly emissions (output * emissions rate)
                    hourly_co2 = (gen_output * emissions).sum(axis=1)
                    total_co2 = (hourly_co2 * snapshot_weights).sum()
                except Exception:
                    total_co2 = np.nan
            else:
                # No carrier-level emissions table available; leave as NaN
                total_co2 = np.nan

    summary['total_co2_emissions'] = total_co2

    # =====================================================================
    # LINE LOADING STATISTICS
    # =====================================================================

    avg_line_loading = np.nan
    max_line_loading = np.nan
    lines_above_60pct = 0
    lines_above_70pct = 0

    if has_lines(network):
        if hasattr(network, 'lines_t') and hasattr(network.lines_t, 'p0'):
            line_power = network.lines_t.p0  # snapshots x lines

            # Calculate loading as |flow| / s_nom
            line_capacity = network.lines['s_nom'].values  # 1D array of capacities

            # Avoid division by zero
            nonzero_capacity = line_capacity > 0

            if nonzero_capacity.sum() > 0:
                # Calculate loading for each snapshot and line
                loading_matrix = np.abs(line_power.values[:, nonzero_capacity]) / line_capacity[nonzero_capacity]

                # Average loading per line across snapshots (weighted by snapshot weights)
                avg_loading_per_line = (loading_matrix.T * snapshot_weights.values).sum(axis=1) / snapshot_weights.sum()

                # Overall statistics
                avg_line_loading = avg_loading_per_line.mean() * 100  # Convert to percentage
                max_line_loading = loading_matrix.max() * 100

                # Count lines above thresholds
                lines_above_60pct = (avg_loading_per_line > 0.60).sum()
                lines_above_70pct = (avg_loading_per_line > 0.70).sum()

    summary['avg_line_loading_pct'] = avg_line_loading
    summary['max_line_loading_pct'] = max_line_loading
    summary['lines_above_60pct'] = int(lines_above_60pct)
    summary['lines_above_70pct'] = int(lines_above_70pct)

    return summary


# Build summary DataFrame from all networks
print("Computing network summaries...")
summary_data = []

for (scenario_name, wind_condition, transmission_setting), network in networks.items():
    try:
        row_summary = compute_network_summary(
            network,
            scenario_name,
            wind_condition,
            transmission_setting
        )
        summary_data.append(row_summary)
    except Exception as e:
        print(f"[ERROR] Failed to compute summary for {scenario_name} / {wind_condition}: {e}")

# Create comprehensive summary DataFrame
if summary_data:
    summary_df = pd.DataFrame(summary_data)
    
    # Reorder columns for readability
    col_order = [
        'scenario', 'wind_condition', 'transmission_setting',
        'load_served_MWh', 'load_shedding_MWh',
        'avg_marginal_price', 'max_marginal_price',
        'total_co2_emissions',
        'avg_line_loading_pct', 'max_line_loading_pct',
        'lines_above_60pct', 'lines_above_70pct'
    ]
    
    # Select only columns that exist
    col_order = [c for c in col_order if c in summary_df.columns]
    summary_df = summary_df[col_order]
    
    print(f"\n[OK] Computed summaries for {len(summary_df)} networks")
    print("\nNetwork Summary DataFrame:")
    print(summary_df.to_string())
else:
    print("[ERROR] No summaries could be computed")
    summary_df = pd.DataFrame()


Computing network summaries...

[OK] Computed summaries for 18 networks

Network Summary DataFrame:
        scenario   wind_condition   transmission_setting  load_served_MWh  load_shedding_MWh  avg_marginal_price  max_marginal_price  total_co2_emissions  avg_line_loading_pct  max_line_loading_pct  lines_above_60pct  lines_above_70pct
0       base-ext         notwindy             extendable     1.300023e+06                NaN           51.843323           59.936031        179982.188307             14.342473            322.222071                  3                  2
1       base-ext  windvariability             extendable     1.368958e+06                NaN           23.045886           35.601855        101139.237744             32.350663            348.067545                 80                 46
2       base-ext            windy             extendable     1.179665e+06                NaN            5.002172           21.994161           108.569104             27.164156            246.0

## Section 8: Line-Level Congestion Statistics

In [10]:
# Use consolidated has_lines helper defined in the utilities cell above

def compute_line_congestion_stats(network: pypsa.Network) -> pd.DataFrame:
    """
    Compute line-level congestion statistics for a solved network.
    
    Parameters
    ----------
    network : pypsa.Network
        Solved PyPSA network
    
    Returns
    -------
    pd.DataFrame
        One row per line with columns:
        - line_id: index name of the line
        - bus0: source bus
        - bus1: target bus
        - average_loading_pct: weighted average loading across all snapshots
        - max_loading_pct: maximum instantaneous loading
        - hours_above_60: number of weighted hours with loading > 60%
        - hours_above_70: number of weighted hours with loading > 70%
    """
    if not has_lines(network):
        return pd.DataFrame()
    
    if not hasattr(network, 'lines_t') or not hasattr(network.lines_t, 'p0'):
        return pd.DataFrame()
    
    # Get line power flows and capacities
    line_flows = network.lines_t.p0  # snapshots x lines
    
    # Determine which capacity to use (optimized or original)
    if 's_nom_opt' in network.lines.columns:
        line_capacity = network.lines['s_nom_opt'].copy()
        # Handle any NaN values in optimized capacity
        line_capacity.fillna(network.lines['s_nom'], inplace=True)
    else:
        line_capacity = network.lines['s_nom'].copy()
    
    # Get snapshot weights (uniform hourly snapshots)
    snapshot_weights = snapshot_hours(network)
    
    # Build results
    results = []
    
    for line_idx in network.lines.index:
        line_row = network.lines.loc[line_idx]
        
        # Get capacity, handle zero or NaN
        capacity = line_capacity.loc[line_idx]
        if np.isnan(capacity) or capacity <= 0:
            # Skip lines with invalid capacity
            continue
        
        # Get power flow for this line
        if line_idx in line_flows.columns:
            flow = line_flows[line_idx].values
        else:
            continue
        
        # Calculate loading as absolute flow / capacity
        loading = np.abs(flow) / capacity
        
        # Calculate statistics
        avg_loading = (loading * snapshot_weights.values).sum() / snapshot_weights.sum()
        max_loading = loading.max()
        
        # Count hours above thresholds
        hours_above_60 = ((loading > 0.60) * snapshot_weights.values).sum()
        hours_above_70 = ((loading > 0.70) * snapshot_weights.values).sum()
        
        results.append({
            'line_id': line_idx,
            'bus0': line_row['bus0'],
            'bus1': line_row['bus1'],
            'average_loading_pct': avg_loading * 100,
            'max_loading_pct': max_loading * 100,
            'hours_above_60': hours_above_60,
            'hours_above_70': hours_above_70,
        })
    
    return pd.DataFrame(results)


def get_top_loaded_lines(
    line_stats: pd.DataFrame,
    n: int = 10,
    sort_by: str = 'average_loading_pct'
) -> pd.DataFrame:
    """
    Get the top N most loaded lines from congestion statistics.
    
    Parameters
    ----------
    line_stats : pd.DataFrame
        Output from compute_line_congestion_stats()
    n : int
        Number of top lines to return (default: 10)
    sort_by : str
        Column to sort by (default: 'average_loading_pct')
        Options: 'average_loading_pct', 'max_loading_pct', 'hours_above_70', etc.
    
    Returns
    -------
    pd.DataFrame
        Top N lines sorted by the specified metric
    """
    if line_stats.empty:
        return pd.DataFrame()
    
    if sort_by not in line_stats.columns:
        print(f"[WARNING] Column '{sort_by}' not found. Using 'average_loading_pct'")
        sort_by = 'average_loading_pct'
    
    return line_stats.nlargest(n, sort_by)


# Build line statistics dictionary for all networks
line_stats_by_run = {}

for (scenario_name, wind_condition, transmission_setting), network in networks.items():
    try:
        line_stats = compute_line_congestion_stats(network)
        line_stats_by_run[(scenario_name, wind_condition, transmission_setting)] = line_stats
        n_congested = len(line_stats)
    except Exception as e:
        print(f"[ERROR] Failed for {scenario_name} / {wind_condition}: {e}")
        line_stats_by_run[(scenario_name, wind_condition, transmission_setting)] = pd.DataFrame()

# Example: Show top 5 most loaded lines from first network
if line_stats_by_run:
    first_key = next(iter(line_stats_by_run.keys()))
    first_stats = line_stats_by_run[first_key]
    
    if not first_stats.empty:
        print(f"\nTop 5 most loaded lines for {first_key}:")
        top_lines = get_top_loaded_lines(first_stats, n=TOP_N_LINES, sort_by='average_loading_pct')
        print(top_lines.to_string(index=False))
    else:
        print(f"No line data available for {first_key}")



Top 5 most loaded lines for ('base-ext', 'notwindy', 'extendable'):
line_id    bus0    bus1  average_loading_pct  max_loading_pct  hours_above_60  hours_above_70
    235  DE0 18 DE0 282            54.488889        69.905040            10.0             0.0
     99  DE0 13  DE0 69            54.292747        69.561454            12.0             0.0
    472 DE0 292 DE0 430            52.268396        69.862517             9.0             0.0
    119 DE0 136 DE0 317            51.166352        69.819695             9.0             0.0
    167 DE0 155 DE0 292            49.706941        69.860980            16.0             0.0


## Section 9: Bus-Level Stress Statistics

In [11]:
def compute_bus_stress_stats(network: pypsa.Network) -> pd.DataFrame:
    """
    Compute bus-level stress statistics for a solved network.

    Only buses with positive load shedding are returned.
    """
    if network.buses.empty:
        return pd.DataFrame()

    marginal_prices = None
    if hasattr(network, 'buses_t') and hasattr(network.buses_t, 'marginal_price'):
        if not network.buses_t.marginal_price.empty:
            marginal_prices = network.buses_t.marginal_price

    if network.generators.empty or 'carrier' not in network.generators.columns:
        return pd.DataFrame()

    load_mask = network.generators.carrier.str.contains(SHEDDING_CARRIER_REGEX, case=False, na=False)
    shed_gens = network.generators.index[load_mask]

    if len(shed_gens) == 0:
        return pd.DataFrame()

    if not hasattr(network, 'generators_t') or not hasattr(network.generators_t, 'p'):
        return pd.DataFrame()

    available_shed_gens = [gen for gen in shed_gens if gen in network.generators_t.p.columns]
    if len(available_shed_gens) == 0:
        return pd.DataFrame()

    shed_output = network.generators_t.p[available_shed_gens].fillna(0)
    shed_bus = network.generators.loc[available_shed_gens, 'bus']
    shed_by_bus = shed_output.groupby(shed_bus, axis=1).sum()

    total_shedding_by_bus = shed_by_bus.sum(axis=0)
    positive_buses = total_shedding_by_bus[total_shedding_by_bus > 0]

    if positive_buses.empty:
        return pd.DataFrame()

    max_shedding_by_bus = shed_by_bus.max(axis=0)
    worst_snapshot_by_bus = shed_by_bus.idxmax(axis=0)

    results = []
    for bus_id, total_load_shedding_mwh in positive_buses.items():
        bus_row = network.buses.loc[bus_id] if bus_id in network.buses.index else pd.Series(dtype=float)

        if marginal_prices is not None and bus_id in marginal_prices.columns:
            prices = marginal_prices[bus_id].dropna()
            average_marginal_price = prices.mean() if not prices.empty else np.nan
            max_marginal_price = prices.max() if not prices.empty else np.nan
        else:
            average_marginal_price = np.nan
            max_marginal_price = np.nan

        results.append({
            'bus_id': bus_id,
            'x': bus_row['x'] if 'x' in bus_row else np.nan,
            'y': bus_row['y'] if 'y' in bus_row else np.nan,
            'average_marginal_price': average_marginal_price,
            'max_marginal_price': max_marginal_price,
            'total_load_shedding_mwh': float(total_load_shedding_mwh),
            'max_load_shedding_mw': float(max_shedding_by_bus.loc[bus_id]),
            'worst_shedding_snapshot': worst_snapshot_by_bus.loc[bus_id],
        })

    bus_stats = pd.DataFrame(results)
    if not bus_stats.empty:
        bus_stats = bus_stats.sort_values('total_load_shedding_mwh', ascending=False).reset_index(drop=True)
    return bus_stats


def get_top_buses_by_price(
    bus_stats: pd.DataFrame,
    n: int = 10,
    price_metric: str = 'average_marginal_price'
) -> pd.DataFrame:
    """
    Get the top N buses by marginal price.
    """
    if bus_stats.empty:
        return pd.DataFrame()

    if price_metric not in bus_stats.columns:
        print(f"[WARNING] Column '{price_metric}' not found. Using 'average_marginal_price'")
        price_metric = 'average_marginal_price'

    valid_data = bus_stats[bus_stats[price_metric].notna()]
    if valid_data.empty:
        return pd.DataFrame()

    return valid_data.nlargest(n, price_metric)


def get_top_buses_by_load_shedding(
    bus_stats: pd.DataFrame,
    n: int = 10,
    shedding_metric: str = 'total_load_shedding_mwh'
) -> pd.DataFrame:
    """
    Get the top N buses by load shedding.
    """
    if bus_stats.empty:
        return pd.DataFrame()

    if shedding_metric not in bus_stats.columns:
        print(f"[WARNING] Column '{shedding_metric}' not found. Using 'total_load_shedding_mwh'")
        shedding_metric = 'total_load_shedding_mwh'

    valid_data = bus_stats[bus_stats[shedding_metric].notna()]
    if valid_data.empty:
        return pd.DataFrame()

    return valid_data.nlargest(n, shedding_metric)


# Build bus statistics dictionary for all networks
print("Computing bus-level stress statistics...")
bus_stats_by_run = {}

for (scenario_name, wind_condition, transmission_setting), network in networks.items():
    try:
        bus_stats = compute_bus_stress_stats(network)
        if bus_stats.empty:
            print(f"[OK] {scenario_name} / {wind_condition}: 0 buses with shedding")
            continue

        bus_stats_by_run[(scenario_name, wind_condition, transmission_setting)] = bus_stats
        print(f"[OK] {scenario_name} / {wind_condition}: {len(bus_stats)} buses with shedding")
    except Exception as e:
        print(f"[ERROR] Failed for {scenario_name} / {wind_condition}: {e}")

print(f"\n[OK] Computed bus statistics for {len(bus_stats_by_run)} runs")

# Example: Show top buses by price and shedding from first network
if bus_stats_by_run:
    first_key = next(iter(bus_stats_by_run.keys()))
    first_stats = bus_stats_by_run[first_key]

    if not first_stats.empty:
        print(f"\n[INFO] Bus stress statistics for {first_key}:")

        top_price = get_top_buses_by_price(first_stats, n=5, price_metric='average_marginal_price')
        if not top_price.empty:
            print("\nTop 5 buses by average marginal price:")
            print(top_price[['bus_id', 'average_marginal_price', 'max_marginal_price']].to_string(index=False))

        top_shedding = get_top_buses_by_load_shedding(first_stats, n=5, shedding_metric='total_load_shedding_mwh')
        if not top_shedding.empty:
            print("\nTop 5 buses by load shedding:")
            print(top_shedding[['bus_id', 'total_load_shedding_mwh', 'max_load_shedding_mw', 'worst_shedding_snapshot']].to_string(index=False))
        else:
            print("\nNo load shedding detected in this network")
    else:
        print(f"No bus data available for {first_key}")


Computing bus-level stress statistics...
[OK] base-ext / notwindy: 1350 buses with shedding
[OK] base-ext / windvariability: 1350 buses with shedding
[OK] base-ext / windy: 1350 buses with shedding
[OK] base2 / notwindy: 1350 buses with shedding
[OK] base2 / windvariability: 1350 buses with shedding
[OK] base2 / windy: 1350 buses with shedding
[OK] baseTP / notwindy: 1350 buses with shedding
[OK] baseTP / windvariability: 1350 buses with shedding
[OK] baseTP / windy: 1350 buses with shedding
[OK] scenario-ext / notwindy: 1350 buses with shedding
[OK] scenario-ext / windvariability: 1350 buses with shedding
[OK] scenario-ext / windy: 1350 buses with shedding
[OK] scenario1 / notwindy: 1350 buses with shedding
[OK] scenario1 / windvariability: 1350 buses with shedding
[OK] scenario1 / windy: 1350 buses with shedding
[OK] scenario2 / notwindy: 1350 buses with shedding
[OK] scenario2 / windvariability: 1350 buses with shedding
[OK] scenario2 / windy: 1350 buses with shedding

[OK] Computed

In [27]:
# Section 9.0: Curtailment calculation by carrier and run
# Builds `curtailment_df` before Section 9.1 uses it.


def compute_curtailment_by_carrier(network: pypsa.Network) -> pd.DataFrame:
    """Compute carrier-level available, dispatched, and curtailed power for one network.

    Uses generator carrier grouping and treats snapshots as uniform hourly samples.
    Returns one row per carrier with run-independent metrics.
    """
    if not has_generators(network):
        return pd.DataFrame()

    if not hasattr(network, 'generators_t') or not hasattr(network.generators_t, 'p'):
        return pd.DataFrame()

    if not hasattr(network.generators_t, 'p_max_pu'):
        return pd.DataFrame()

    gens = network.generators
    p_max_pu = network.generators_t.p_max_pu
    p = network.generators_t.p

    available_gens = [g for g in gens.index if g in p_max_pu.columns and g in p.columns]
    if not available_gens:
        return pd.DataFrame()

    gens = gens.loc[available_gens]
    p_max_pu = p_max_pu[available_gens].fillna(0)
    p = p[available_gens].fillna(0)

    # Instantaneous available power (MW) and dispatched power (MW)
    p_available = p_max_pu.multiply(gens['p_nom'], axis=1)
    p_available_by_carrier = p_available.T.groupby(gens['carrier']).sum().T
    p_by_carrier = p.T.groupby(gens['carrier']).sum().T
    p_curtailed_by_carrier = (p_available_by_carrier - p_by_carrier).clip(lower=0)

    # Convert snapshots to hours (uniform 1h snapshots by default)
    hours = snapshot_hours(network)
    hours = hours.reindex(p_available_by_carrier.index).fillna(SNAPSHOT_HOURS)

    rows = []
    for carrier in p_available_by_carrier.columns:
        available_ts = p_available_by_carrier[carrier]
        dispatched_ts = p_by_carrier[carrier]
        curtailed_ts = p_curtailed_by_carrier[carrier]

        total_available = float((available_ts * hours).sum())
        total_dispatched = float((dispatched_ts * hours).sum())
        total_curtailed = float((curtailed_ts * hours).sum())
        capacity = float(gens.groupby('carrier')['p_nom'].sum().get(carrier, np.nan))

        rows.append({
            'carrier': carrier,
            'available_mwh': total_available,
            'dispatched_mwh': total_dispatched,
            'curtailed_mwh': total_curtailed,
            'capacity_mw': capacity,
            'curtailment_share_pct': (total_curtailed / total_available * 100.0) if total_available > 0 else np.nan,
        })

    curtailment = pd.DataFrame(rows)
    if not curtailment.empty:
        curtailment = curtailment.sort_values('curtailed_mwh', ascending=False).reset_index(drop=True)
    return curtailment


def build_curtailment_dataframe(networks: Dict[Tuple[str, str, str], pypsa.Network]) -> pd.DataFrame:
    """Build a long-format curtailment dataframe across all runs."""
    rows = []
    for (scenario, wind_condition, transmission_setting), network in networks.items():
        curtailment = compute_curtailment_by_carrier(network)
        if curtailment.empty:
            continue

        curtailment = curtailment.copy()
        curtailment['scenario'] = scenario
        curtailment['wind_condition'] = wind_condition
        curtailment['transmission_setting'] = transmission_setting
        
        rows.append(curtailment[curtailment['curtailment_share_pct'].notnull()])

    if not rows:
        return pd.DataFrame()

    curtailment_df = pd.concat(rows, ignore_index=True)
    # Put run identifiers first for readability
    run_cols = ['scenario', 'wind_condition', 'transmission_setting']
    other_cols = [c for c in curtailment_df.columns if c not in run_cols]
    return curtailment_df[run_cols + other_cols]


curtailment_df = build_curtailment_dataframe(networks)
if not curtailment_df.empty:
    print(f"Curtailment rows: {len(curtailment_df)}")
    display(curtailment_df.head(10))
else:
    print("No curtailment data found in the loaded networks.")

Curtailment rows: 72


,scenario,wind_condition,transmission_setting,carrier,available_mwh,dispatched_mwh,curtailed_mwh,capacity_mw,curtailment_share_pct
0,base-ext,notwindy,extendable,solar,4.381812e+05,4.381807e+05,0.512230,86408.000000,0.000117
1,base-ext,notwindy,extendable,ror,6.387188e+04,6.387147e+04,0.410235,4014.118709,0.000642
2,base-ext,notwindy,extendable,onwind,4.943540e+04,4.943508e+04,0.316428,73332.065900,0.000640
3,base-ext,notwindy,extendable,offwind-ac,8.316630e+03,8.316625e+03,0.004177,11163.743000,0.000050
4,base-ext,windvariability,extendable,onwind,8.048308e+05,7.725724e+05,32258.431284,73332.065900,4.008101
5,base-ext,windvariability,extendable,offwind-ac,1.735688e+05,1.596849e+05,13883.821028,11163.743000,7.999032
6,base-ext,windvariability,extendable,ror,2.621926e+04,2.620386e+04,15.398738,4014.118709,0.058731
7,base-ext,windvariability,extendable,solar,9.012128e+04,9.011739e+04,3.883319,86408.000000,0.004309
8,base-ext,windy,extendable,onwind,1.520776e+06,1.007248e+06,513528.213103,73332.065900,33.767507
9,base-ext,windy,extendable,offwind-ac,2.362175e+05,6.535762e+04,170859.864363,11163.743000,72.331592


## Section 9.1: Final comparison tables (run-level and long-format aggregations)
Creates: `summary_df` (ensures), `curtailment_report` (run-aggregated), `top_lines_all_runs_df` (long-format top congested lines), `top_buses_all_runs_df` (long-format top stressed buses)

In [14]:
from typing import List

# helper to ensure run columns exist on a dataframe
def _ensure_run_cols(df: pd.DataFrame, scenario, wind, trans) -> pd.DataFrame:
    df = df.copy()
    if 'scenario' not in df.columns:
        df['scenario'] = scenario
    if 'wind_condition' not in df.columns:
        df['wind_condition'] = wind
    if 'transmission_setting' not in df.columns:
        df['transmission_setting'] = trans
    # keep column order common for readability
    run_cols = ['scenario', 'wind_condition', 'transmission_setting']
    other_cols = [c for c in df.columns if c not in run_cols]
    return df[run_cols + other_cols]

# 1) summary_df (ensure it exists and contains run identifiers)
if 'summary_df' in globals():
    summary_df = summary_df.copy()
else:
    if 'summary_data' in globals() and summary_data:
        summary_df = pd.DataFrame(summary_data)
    else:
        summary_df = pd.DataFrame()

for col in ['scenario', 'wind_condition', 'transmission_setting']:
    if col not in summary_df.columns:
        summary_df[col] = np.nan

# 2) curtailment_report: aggregate numeric curtailment metrics per run
curtailment_report = pd.DataFrame()
if 'curtailment_df' in globals():
    curtailment_report = curtailment_df.copy()
    # Ensure run cols exist (if individual rows already include them they are kept)
    for col in ['scenario', 'wind_condition', 'transmission_setting']:
        if col not in curtailment_report.columns:
            curtailment_report[col] = np.nan

    # Aggregate numeric metrics per run
    numeric = curtailment_report.select_dtypes(include=[np.number]).columns.tolist()
    if numeric:
        agg = curtailment_report.groupby(['scenario', 'wind_condition', 'transmission_setting'])[numeric].sum().reset_index()
        curtailment_report = agg
        # Create a pivot (scenario x transmission_setting with wind_condition as columns) for quick comparison if possible
    else:
        # no numeric columns to aggregate; keep original
        curtailment_report = curtailment_report

# 3) long-format dataframe of top congested lines across all runs
top_lines_all_runs_df = pd.DataFrame()
_line_rows: List[pd.DataFrame] = []
if 'line_stats_by_run' in globals():
    for (scenario, wind, trans), df in line_stats_by_run.items():
        if df is None or df.empty:
            continue
        tmp = _ensure_run_cols(df, scenario, wind, trans)
        # choose sorting column
        sort_col = 'average_loading_pct' if 'average_loading_pct' in tmp.columns else ('max_loading_pct' if 'max_loading_pct' in tmp.columns else None)
        if sort_col is not None:
            try:
                top = tmp.nlargest(TOP_N_LINES, sort_col)
            except Exception:
                top = tmp.sort_values(sort_col, ascending=False).head(TOP_N_LINES)
        else:
            top = tmp.head(TOP_N_LINES)
        _line_rows.append(top)
    if _line_rows:
        top_lines_all_runs_df = pd.concat(_line_rows, ignore_index=True)

# 4) long-format dataframe of top stressed buses across all runs
top_buses_all_runs_df = pd.DataFrame()
_bus_rows: List[pd.DataFrame] = []
if 'bus_stats_by_run' in globals():
    for (scenario, wind, trans), df in bus_stats_by_run.items():
        if df is None or df.empty:
            continue
        tmp = _ensure_run_cols(df, scenario, wind, trans)
        sort_col = 'total_load_shedding_mwh' if 'total_load_shedding_mwh' in tmp.columns else ('average_marginal_price' if 'average_marginal_price' in tmp.columns else None)
        if sort_col is not None:
            try:
                topb = tmp.nlargest(TOP_N_BUSES, sort_col)
            except Exception:
                topb = tmp.sort_values(sort_col, ascending=False).head(TOP_N_BUSES)
        else:
            topb = tmp.head(TOP_N_BUSES)
        _bus_rows.append(topb)
    if _bus_rows:
        top_buses_all_runs_df = pd.concat(_bus_rows, ignore_index=True)

# Make outputs easy to inspect and export
# expose variables with stable names the rest of the notebook expects
final_summary_df = summary_df
final_curtailment_report = curtailment_report
final_top_lines = top_lines_all_runs_df
final_top_buses = top_buses_all_runs_df

print('Final comparison tables created:')
print(' - final_summary_df:', final_summary_df.shape)
print(' - final_curtailment_report:', final_curtailment_report.shape)
print(' - final_top_lines:', final_top_lines.shape)
print(' - final_top_buses:', final_top_buses.shape)

# Display small previews for quick inspection
if not final_summary_df.empty:
    print('\nSummary (first 5 rows):')
    display(final_summary_df.head())
if not final_curtailment_report.empty:
    print('\nCurtailment report (first 5 rows):')
    display(final_curtailment_report.head())
if not final_top_lines.empty:
    print('\nTop congested lines (first 10 rows):')
    display(final_top_lines.head(10))
if not final_top_buses.empty:
    print('\nTop stressed buses (first 10 rows):')
    display(final_top_buses.head(10))


Final comparison tables created:
 - final_summary_df: (18, 12)
 - final_curtailment_report: (18, 8)
 - final_curtailment_pivot: (6, 17)
 - final_top_lines: (90, 10)
 - final_top_buses: (90, 11)

Summary (first 5 rows):


,scenario,wind_condition,transmission_setting,load_served_MWh,load_shedding_MWh,avg_marginal_price,max_marginal_price,total_co2_emissions,avg_line_loading_pct,max_line_loading_pct,lines_above_60pct,lines_above_70pct
0,base-ext,notwindy,extendable,1.300023e+06,NaN,51.843323,59.936031,179982.188307,14.342473,322.222071,3,2
1,base-ext,windvariability,extendable,1.368958e+06,NaN,23.045886,35.601855,101139.237744,32.350663,348.067545,80,46
2,base-ext,windy,extendable,1.179665e+06,NaN,5.002172,21.994161,108.569104,27.164156,246.032949,34,19
3,base2,notwindy,fixed,1.300023e+06,NaN,54.523992,143.429463,179627.348282,11.849893,69.135637,0,0
4,base2,windvariability,fixed,1.368958e+06,NaN,41.099507,1112.255027,162112.501269,17.016474,69.135542,1,0



Curtailment report (first 5 rows):


,scenario,wind_condition,transmission_setting,available_mwh,dispatched_mwh,curtailed_mwh,capacity_mw,curtailment_share_pct
0,base-ext,notwindy,extendable,5.598051e+05,5.598038e+05,1.243070,174917.927609,0.001449
1,base-ext,windvariability,extendable,1.094740e+06,1.048579e+06,46161.534369,174917.927609,12.070173
2,base-ext,windy,extendable,1.918853e+06,1.233671e+06,685181.867025,174917.927609,106.892361
3,base2,notwindy,fixed,5.598051e+05,5.596700e+05,135.066432,174917.927609,0.034666
4,base2,windvariability,fixed,1.094740e+06,8.067463e+05,287993.833791,174917.927609,89.700196



Top congested lines (first 10 rows):


,scenario,wind_condition,transmission_setting,line_id,bus0,bus1,average_loading_pct,max_loading_pct,hours_above_60,hours_above_70
0,base-ext,notwindy,extendable,235,DE0 18,DE0 282,54.488889,69.905040,10.0,0.0
1,base-ext,notwindy,extendable,99,DE0 13,DE0 69,54.292747,69.561454,12.0,0.0
2,base-ext,notwindy,extendable,472,DE0 292,DE0 430,52.268396,69.862517,9.0,0.0
3,base-ext,notwindy,extendable,119,DE0 136,DE0 317,51.166352,69.819695,9.0,0.0
4,base-ext,notwindy,extendable,167,DE0 155,DE0 292,49.706941,69.860980,16.0,0.0
5,base-ext,windvariability,extendable,99,DE0 13,DE0 69,60.651338,69.535991,14.0,0.0
6,base-ext,windvariability,extendable,173,DE0 158,DE0 389,60.097338,69.348948,15.0,0.0
7,base-ext,windvariability,extendable,235,DE0 18,DE0 282,59.664681,69.860249,15.0,0.0
8,base-ext,windvariability,extendable,449,DE0 282,DE0 365,59.158804,69.880116,15.0,0.0
9,base-ext,windvariability,extendable,560,DE0 365,DE0 389,58.071312,69.490355,14.0,0.0



Top stressed buses (first 10 rows):


,scenario,wind_condition,transmission_setting,bus_id,x,y,average_marginal_price,max_marginal_price,total_load_shedding_mwh,max_load_shedding_mw,worst_shedding_snapshot
0,base-ext,notwindy,extendable,DE0 143 H2,7.985594,50.841773,59.935996,59.936031,0.000009,3.794081e-07,2025-04-08 16:00:00
1,base-ext,notwindy,extendable,DE0 431 H2,8.043546,50.785155,59.896023,59.896057,0.000009,3.794068e-07,2025-04-08 06:00:00
2,base-ext,notwindy,extendable,DE0 373 H2,7.941608,50.847477,59.898207,59.898241,0.000009,3.794067e-07,2025-04-08 15:00:00
3,base-ext,notwindy,extendable,DE0 332 H2,8.044498,50.763664,59.889740,59.889775,0.000009,3.794066e-07,2025-04-08 07:00:00
4,base-ext,notwindy,extendable,DE0 89 H2,8.932898,52.193129,59.883169,59.883206,0.000009,3.794062e-07,2025-04-08 22:00:00
5,base-ext,windvariability,extendable,DE0 47 H2,10.936681,48.401343,35.601821,35.601855,0.000002,8.838691e-08,2025-10-06 17:00:00
6,base-ext,windvariability,extendable,DE0 45 H2,9.012847,47.844702,35.589234,35.589261,0.000002,8.838680e-08,2025-10-06 18:00:00
7,base-ext,windvariability,extendable,DE0 164 H2,10.840260,48.237799,35.564295,35.564329,0.000002,8.838659e-08,2025-10-06 17:00:00
8,base-ext,windvariability,extendable,DE0 78 H2,10.702777,47.826434,35.487140,35.487173,0.000002,8.838592e-08,2025-10-06 17:00:00
9,base-ext,windvariability,extendable,DE0 180 H2,10.863452,48.546958,35.484520,35.484553,0.000002,8.838590e-08,2025-10-06 17:00:00


## Section 10: Cross-Network Bottleneck Analysis

In [15]:
def build_cross_network_bottlenecks(
    line_stats_by_run: Dict,
    networks: Dict[Tuple[str, str, str], pypsa.Network],
    congestion_threshold: float = CONGESTION_THRESHOLD
) -> pd.DataFrame:
    """
    Identify bottleneck lines across multiple network runs.
    
    Parameters
    ----------
    line_stats_by_run : Dict
        Dictionary with keys (scenario, wind_condition, transmission_setting)
        and values as DataFrames from compute_line_congestion_stats()
    networks : Dict[Tuple[str, str, str], pypsa.Network]
        Dictionary of PyPSA networks
    congestion_threshold : float
        Loading percentage threshold to identify congestion (default: 60.0)
    
    Returns
    -------
    pd.DataFrame
        Cross-network bottleneck summary with columns:
        - line_id, bus0, bus1
        - number_of_networks_congested
        - list_of_networks_congested
        - average_loading_across_networks
        - maximum_loading_across_networks
        - loading_change_vs_fixed_case (if fixed case available)
        - line_length, x_coordinate, y_coordinate (if available)
        - carrier (if available)
        - region (if available)
    
    Notes
    -----
    The "fixed case" is identified as the reference scenario with transmission_setting='fixed'.
    Congestion is determined per run, then aggregated across runs.
    """
    
    # Helper function to create readable run name
    def make_run_name(scenario: str, wind: str, transmission: str) -> str:
        return f"{scenario}_{wind}_{transmission}"
    
    # Collect congested lines from each run
    all_congested = []
    
    for (scenario, wind, transmission), line_stats in line_stats_by_run.items():
        if line_stats.empty:
            continue
        
        run_name = make_run_name(scenario, wind, transmission)
        
        # Find congested lines in this run
        congested = line_stats[line_stats['average_loading_pct'] >= congestion_threshold].copy()
        
        if len(congested) > 0:
            congested['run_name'] = run_name
            congested['scenario'] = scenario
            congested['wind_condition'] = wind
            congested['transmission_setting'] = transmission
            all_congested.append(congested)
    
    if not all_congested:
        print("[WARNING] No congested lines found at this threshold")
        return pd.DataFrame()
    
    # Combine all congested lines
    combined = pd.concat(all_congested, ignore_index=True)
    
    # Group by line_id and aggregate
    bottlenecks = []
    
    for line_id in combined['line_id'].unique():
        line_data = combined[combined['line_id'] == line_id]
        
        bottleneck = {
            'line_id': line_id,
            'bus0': line_data['bus0'].iloc[0],
            'bus1': line_data['bus1'].iloc[0],
            'number_of_networks_congested': len(line_data),
            'list_of_networks_congested': ', '.join(sorted(line_data['run_name'].unique())),
            'average_loading_across_networks': line_data['average_loading_pct'].mean(),
            'maximum_loading_across_networks': line_data['average_loading_pct'].max(),
            'min_loading_across_networks': line_data['average_loading_pct'].min(),
        }
        
        # Find loading change vs fixed case
        fixed_case_loadings = line_data[
            line_data['transmission_setting'] == 'fixed'
        ]['average_loading_pct']
        
        if len(fixed_case_loadings) > 0:
            fixed_loading = fixed_case_loadings.iloc[0]
            other_loadings = line_data[
                line_data['transmission_setting'] != 'fixed'
            ]['average_loading_pct']
            
            if len(other_loadings) > 0:
                bottleneck['loading_change_vs_fixed'] = other_loadings.mean() - fixed_loading
            else:
                bottleneck['loading_change_vs_fixed'] = np.nan
        else:
            bottleneck['loading_change_vs_fixed'] = np.nan
        
        bottlenecks.append(bottleneck)
    
    bottleneck_df = pd.DataFrame(bottlenecks)
    
    # =====================================================================
    # ADD ADDITIONAL COLUMNS FROM NETWORK DATA
    # =====================================================================
    
    # Get reference network (first available) to extract line metadata
    if networks:
        ref_network = next(iter(networks.values()))
        
        # Add line metadata if available
        if not ref_network.lines.empty:
            line_meta = {}
            
            for line_id in ref_network.lines.index:
                meta = {
                    'line_length': np.nan,
                    'x_coordinate': np.nan,
                    'y_coordinate': np.nan,
                    'carrier': 'AC',
                }
                
                # Try to extract length
                if 'length' in ref_network.lines.columns:
                    meta['line_length'] = ref_network.lines.loc[line_id, 'length']
                
                # Try to extract carrier
                if 'carrier' in ref_network.lines.columns:
                    meta['carrier'] = ref_network.lines.loc[line_id, 'carrier']
                
                # Try to get bus coordinates
                try:
                    bus0_idx = ref_network.lines.loc[line_id, 'bus0']
                    bus1_idx = ref_network.lines.loc[line_id, 'bus1']
                    
                    if bus0_idx in ref_network.buses.index:
                        bus0 = ref_network.buses.loc[bus0_idx]
                        if 'x' in bus0 and 'y' in bus0:
                            meta['bus0_x'] = bus0['x']
                            meta['bus0_y'] = bus0['y']
                        if 'country' in bus0:
                            meta['country'] = bus0['country']
                except Exception:
                    pass
                
                line_meta[line_id] = meta
            
            # Merge metadata into bottleneck dataframe
            meta_df = pd.DataFrame(line_meta).T
            bottleneck_df = bottleneck_df.merge(
                meta_df,
                left_on='line_id',
                right_index=True,
                how='left'
            )
    
    # =====================================================================
    # SORT AND REORDER COLUMNS FOR REPORT
    # =====================================================================
    
    # Sort by number of networks (desc) then average loading (desc)
    bottleneck_df = bottleneck_df.sort_values(
        by=['number_of_networks_congested', 'average_loading_across_networks'],
        ascending=[False, False]
    ).reset_index(drop=True)
    
    # Reorder columns for readability
    priority_cols = [
        'line_id', 'bus0', 'bus1',
        'number_of_networks_congested',
        'list_of_networks_congested',
        'average_loading_across_networks',
        'maximum_loading_across_networks',
        'min_loading_across_networks',
        'loading_change_vs_fixed',
    ]
    
    optional_cols = [
        'line_length', 'carrier',
        'bus0_x', 'bus0_y',
        'country', 'x_coordinate', 'y_coordinate'
    ]
    
    # Select columns that exist
    col_order = [c for c in priority_cols if c in bottleneck_df.columns]
    col_order += [c for c in optional_cols if c in bottleneck_df.columns]
    col_order += [c for c in bottleneck_df.columns if c not in col_order]
    
    bottleneck_df = bottleneck_df[col_order]
    
    # Round numerical columns for readability
    numeric_cols = bottleneck_df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if 'pct' in col or 'loading' in col:
            bottleneck_df[col] = bottleneck_df[col].round(2)
        elif 'length' in col or 'coordinate' in col:
            bottleneck_df[col] = bottleneck_df[col].round(4)
    
    return bottleneck_df


# Build cross-network bottleneck table
print("Building cross-network bottleneck table...")
print(f"Congestion threshold: {CONGESTION_THRESHOLD}%")

bottleneck_df = build_cross_network_bottlenecks(
    line_stats_by_run,
    networks,
    congestion_threshold=CONGESTION_THRESHOLD
)

if not bottleneck_df.empty:
    print(f"\n[OK] Identified {len(bottleneck_df)} bottleneck lines across networks")
    print("\nCross-Network Bottleneck Summary:")
    print(bottleneck_df.to_string(index=True))
    
    # Summary statistics
    print("\n" + "=" * 70)
    print("BOTTLENECK STATISTICS")
    print("=" * 70)
    print(f"Total unique bottleneck lines: {len(bottleneck_df)}")
    print(f"Lines in 2+ networks: {(bottleneck_df['number_of_networks_congested'] >= 2).sum()}")
    print(f"Average congestion across bottlenecks: {bottleneck_df['average_loading_across_networks'].mean():.1f}%")
    print(f"Most severely congested line: {bottleneck_df['line_id'].iloc[0]} ({bottleneck_df['average_loading_across_networks'].iloc[0]:.1f}%)")
else:
    print("[WARNING] No bottleneck lines identified")
    bottleneck_df = pd.DataFrame()


Building cross-network bottleneck table...
Congestion threshold: 60.0%

[OK] Identified 97 bottleneck lines across networks

Cross-Network Bottleneck Summary:
   line_id     bus0     bus1  number_of_networks_congested                                                                                                                                                                                list_of_networks_congested  average_loading_across_networks  maximum_loading_across_networks  min_loading_across_networks  loading_change_vs_fixed line_length carrier     bus0_x     bus0_y country x_coordinate y_coordinate
0       99   DE0 13   DE0 69                             7  base-ext_windvariability_extendable, base-ext_windy_extendable, scenario1_windvariability_fixed, scenario1_windy_fixed, scenario2_notwindy_fixed, scenario2_windvariability_fixed, scenario2_windy_fixed                            65.72                            69.78                        60.65                    -7.63   2

In [16]:
def export_bottleneck_report(
    bottleneck_df: pd.DataFrame,
    output_path: Optional[str] = None,
    min_networks: int = 1,
    min_loading: float = CONGESTION_THRESHOLD
) -> pd.DataFrame:
    """
    Filter and export bottleneck report to CSV.
    
    Parameters
    ----------
    bottleneck_df : pd.DataFrame
        Output from build_cross_network_bottlenecks()
    output_path : str, optional
        Path to save CSV file. If None, no file is written.
    min_networks : int
        Minimum number of networks where line must be congested (default: 1)
    min_loading : float
        Minimum average loading percentage (default: 60.0)
    
    Returns
    -------
    pd.DataFrame
        Filtered bottleneck dataframe
    """
    if bottleneck_df.empty:
        print("[WARNING] Bottleneck dataframe is empty")
        return pd.DataFrame()
    
    # Apply filters
    filtered = bottleneck_df[
        (bottleneck_df['number_of_networks_congested'] >= min_networks) &
        (bottleneck_df['average_loading_across_networks'] >= min_loading)
    ].copy()
    
    if filtered.empty:
        print(f"[WARNING] No bottlenecks found with min_networks={min_networks} and min_loading={min_loading}%")
        return filtered
    
    print(f"[OK] Filtered to {len(filtered)} bottleneck lines")
    
    # Export to CSV if path provided
    if output_path is not None:
        try:
            filtered.to_csv(output_path, index=True)
            print(f"[OK] Exported to: {output_path}")
        except Exception as e:
            print(f"[ERROR] Failed to export: {e}")
    
    return filtered


# Example: Export with different filters
print("\n" + "=" * 70)
print("BOTTLENECK REPORT EXPORT")
print("=" * 70)

# Export all bottlenecks
bottleneck_report = export_bottleneck_report(
    bottleneck_df,
    output_path=None,  # Change to "bottleneck_report.csv" to save
    min_networks=1,
    min_loading=CONGESTION_THRESHOLD
)

# Show lines appearing in multiple networks
if not bottleneck_df.empty:
    multi_network = bottleneck_df[bottleneck_df['number_of_networks_congested'] >= 2]
    if not multi_network.empty:
        print(f"\nLines appearing in 2+ networks ({len(multi_network)} lines):")
        print(multi_network[['line_id', 'bus0', 'bus1', 'number_of_networks_congested', 
                            'average_loading_across_networks', 'loading_change_vs_fixed']].to_string(index=False))



BOTTLENECK REPORT EXPORT
[OK] Filtered to 97 bottleneck lines

Lines appearing in 2+ networks (43 lines):
line_id    bus0    bus1  number_of_networks_congested  average_loading_across_networks  loading_change_vs_fixed
     99  DE0 13  DE0 69                             7                            65.72                    -7.63
    520 DE0 330 DE0 362                             6                            69.16                    -4.47
    622  DE0 61  DE0 96                             6                            68.04                     2.24
    119 DE0 136 DE0 317                             5                            69.51                    -1.01
    178  DE0 16 DE0 304                             5                            68.08                    -2.05
     10   DE0 1 DE0 150                             5                            67.84                    -7.74
     60 DE0 115  DE0 46                             5                            67.67                    -9.

In [18]:
# Simplified export of final tables to CSV using OUTPUT_DIR and final_* variables
from pathlib import Path

OUTPUT_DIR = Path(OUTPUT_DIR) if 'OUTPUT_DIR' in globals() else Path("/home/lucakristin/Desktop/my_pypsa/pypsa-eur/summary_csv")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

exported = []

# Central helper to save DataFrames
def _save_df_minimal(df, filename):
    path = OUTPUT_DIR / filename
    try:
        if df is None or (hasattr(df, 'empty') and df.empty):
            return False
        # prefer index=False for tidy CSVs; include index if explicitly needed later
        df.to_csv(path, index=False)
        exported.append(str(path))
        return True
    except Exception as e:
        print(f"[ERROR] Failed to export {filename}: {e}")
        return False

# Map the canonical export names to variables and filenames
exports = [
    ('summary', globals().get('final_summary_df', globals().get('summary_df', pd.DataFrame())) , 'summary_df.csv'),
    ('curtailment_report', globals().get('final_curtailment_report', globals().get('curtailment_report', pd.DataFrame())), 'curtailment_report.csv'),
    ('curtailment_df', globals().get('curtailment_df', pd.DataFrame()), 'curtailment_df.csv'),
    ('top_lines', globals().get('final_top_lines', globals().get('top_lines_all_runs_df', pd.DataFrame())), 'top_lines_all_runs.csv'),
    ('top_buses', globals().get('final_top_buses', globals().get('top_buses_all_runs_df', pd.DataFrame())), 'top_buses_all_runs.csv'),
    ('bottlenecks', globals().get('bottleneck_df', pd.DataFrame()), 'bottlenecks.csv')
]

for name, df, fname in exports:
    saved = _save_df_minimal(df, fname)
    if saved and VERBOSE:
        print(f"[OK] Exported {name} -> {fname}")
    elif VERBOSE and (df is None or (hasattr(df, 'empty') and df.empty)):
        print(f"[SKIP] {name} is missing or empty")

print('\nExport summary:')
for p in exported:
    print(' -', p)

print('\nDone.')

[OK] Exported summary -> summary_df.csv
[OK] Exported curtailment_report -> curtailment_report.csv
[OK] Exported curtailment_df -> curtailment_df.csv
[OK] Exported top_lines -> top_lines_all_runs.csv
[OK] Exported top_buses -> top_buses_all_runs.csv
[OK] Exported bottlenecks -> bottlenecks.csv

Export summary:
 - /home/lucakristin/Desktop/my_pypsa/pypsa-eur/summary_csv/summary_df.csv
 - /home/lucakristin/Desktop/my_pypsa/pypsa-eur/summary_csv/curtailment_report.csv
 - /home/lucakristin/Desktop/my_pypsa/pypsa-eur/summary_csv/curtailment_df.csv
 - /home/lucakristin/Desktop/my_pypsa/pypsa-eur/summary_csv/top_lines_all_runs.csv
 - /home/lucakristin/Desktop/my_pypsa/pypsa-eur/summary_csv/top_buses_all_runs.csv
 - /home/lucakristin/Desktop/my_pypsa/pypsa-eur/summary_csv/bottlenecks.csv

Done.
